<a href="https://colab.research.google.com/github/ky106031/bert-study/blob/main/251201_%E7%94%9F%E6%88%90AI_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## テキストの類似度を評価する

[Hugging Face](https://huggingface.co/) で公開されている言語モデルを使ってテキストの埋め込みを取得し、コサイン類似度を計算してテキスト間の類似度を評価する。

In [ ]:
# 必要なものを読み込んだりインストールしたりしておく
from sentence_transformers import SentenceTransformer
!pip install fugashi unidic_lite # 一部のモデルで必要

In [ ]:
# 埋め込みモデルを読み込む
# Hugging Face で公開されている埋め込みのモデルから適当にいくつかピックアップした。
#（ただし、SentenceTransformers ライブラリのAPIを持っているもの、非AMBER系モデルに限る。）
# 好きなものを model に読み込んでみよう。

## Ruri-v3-130m
#model = SentenceTransformer("cl-nagoya/ruri-v3-130m")

## Sarashina-Embedding-v2-1B
#model = SentenceTransformer("sbintuitions/sarashina-embedding-v2-1b")

## Sarashina-Embedding-v1-1B
#model = SentenceTransformer("sbintuitions/sarashina-embedding-v1-1b")

## RoSEtta
#model = SentenceTransformer("pkshatech/RoSEtta-base-ja", trust_remote_code=True)

## GLuCoSE v2
#model = SentenceTransformer("pkshatech/GLuCoSE-base-ja-v2")

## Ruri-large-v2
#model = SentenceTransformer("cl-nagoya/ruri-large-v2")

## Japanese SimCSE
#model = SentenceTransformer("cl-nagoya/sup-simcse-ja-large")

In [ ]:
# テキストを入力
sentences = ['私はきつねうどんが好きです',
             '私はうどんが好きなきつねです',
             '私はきつねが好きなうどんです',
             '私が好きなのはきつねうどんです']

In [ ]:
# 埋め込みベクトルを取得
embeddings = model.encode(sentences, convert_to_tensor=True)
print(embeddings)
print(embeddings.shape)

In [ ]:
# いったんデータフレームに置き換える
import pandas as pd
import numpy as np
df_embeddings = pd.DataFrame(embeddings.numpy())
df_embeddings

In [ ]:
# コサイン類似度を計算
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(df_embeddings)
df_similarity_matrix = pd.DataFrame(similarity_matrix, index=sentences, columns=sentences)
df_similarity_matrix

## 性格検査の質問文を可視化

In [ ]:
# 性格特性語と質問文の準備
text_label = ['外向性', '開放性', '協調性', '勤勉性', '神経症傾向']
text_item = [
    '活発で、外向的だと思う',
    '新しいことが好きで、変わった考えをもつと思う',
    '人に気をつかう、やさしい人間だと思う',
    'しっかりしていて、自分に厳しいと思う',
    '心配性で、うろたえやすいと思う'
]

In [ ]:
# 埋め込みを取得してコサイン類似度を計算
embedding_label = model.encode(text_label, convert_to_tensor=True)
embedding_item = model.encode(text_item, convert_to_tensor=True)
df_embedding_label = pd.DataFrame(embedding_label.numpy())
df_embedding_item = pd.DataFrame(embedding_item.numpy())
similarity_matrix = cosine_similarity(df_embedding_item, df_embedding_label)
df_similarity_matrix = pd.DataFrame(similarity_matrix, index=text_item, columns=text_label)
df_similarity_matrix

In [ ]:
# 行ごとにハードマックス関数を適用する
def hardmax(row):
    max_index = row.idxmax()
    return (row == row[max_index]).astype(int)

df_similarity_hardmax = df_similarity_matrix.apply(hardmax, axis=1)
df_similarity_hardmax

In [ ]:
# 主成分分析を実行
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale

pca = PCA(n_components=2)
result_pca = pca.fit_transform(df_similarity_matrix)

# 主成分得点
df_rows = pd.DataFrame(result_pca, index=text_item, columns=["PC1", "PC2"])
df_rows = df_rows.apply(lambda x: (x - x.mean()) / x.std(), axis=0)

# 負荷量
df_cols = pd.DataFrame(pca.components_.T, index=text_label, columns=["PC1", "PC2"])
df_cols = df_cols.apply(lambda x: (x - x.mean()) / x.std(), axis=0)

print(df_rows)
print(df_cols)

In [ ]:
# 主成分分析の結果をバイプロットで可視化
!pip install japanize-matplotlib
import japanize_matplotlib
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_rows["PC1"], df_rows["PC2"])
for i, txt in enumerate(df_rows.index):
    ax.annotate(txt, (df_rows["PC1"][i], df_rows["PC2"][i]), fontsize = 12, ha ="center")
ax.scatter(df_cols["PC1"], df_cols["PC2"])
for i, txt in enumerate(df_cols.index):
    ax.annotate(txt, (df_cols["PC1"][i], df_cols["PC2"][i]), fontsize = 12, ha ="center")

plt.show()

In [ ]:
# 対応分析を実行
!pip install mca
import mca
result_mca = mca.MCA(df_similarity_matrix, benzecri=False)

rows = result_mca.fs_r(N=2)
cols = result_mca.fs_c(N=2)

df_rows = pd.DataFrame(rows, index=text_label, columns=["PC1", "PC2"])
df_cols = pd.DataFrame(cols, index=text_item, columns=["PC1", "PC2"])

print(df_rows)
print(df_cols)

In [ ]:
# 対応分析の結果をバイプロットで可視化
!pip install japanize-matplotlib
import japanize_matplotlib
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_rows["PC1"], df_rows["PC2"])
for i, txt in enumerate(df_rows.index):
    ax.annotate(txt, (df_rows["PC1"][i], df_rows["PC2"][i]), fontsize = 12, ha ="center")
ax.scatter(df_cols["PC1"], df_cols["PC2"])
for i, txt in enumerate(df_cols.index):
    ax.annotate(txt, (df_cols["PC1"][i], df_cols["PC2"][i]), fontsize = 12, ha ="center")

plt.show()

## アンケートの自由記述の中身を可視化

回答データは "FD.csv" として用意してある。

In [ ]:
# アンケートデータの読み込み
# これを実行して、[ファイル選択] で "FD.csv" を選択するとファイルがアップロードされる
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("FD.csv", header=None, names=["opinions"])
df.head()

In [ ]:
# 形態素解析を適用して名詞のみを抽出する
!pip install janome
from janome.tokenizer import Tokenizer

t = Tokenizer()
def extract_nouns(text, tokenizer):
    words = [token.surface for token in tokenizer.tokenize(text) if token.part_of_speech.startswith('名詞')]
    return ' '.join(words)

df['tokens'] = df['opinions'].apply(lambda x: extract_nouns(x, t))
df['tokens'].head()

In [ ]:
# TF-IDF ベクトル化を行ってキーワードを抽出
from sklearn.feature_extraction.text import TfidfVectorizer
custom_stop_words = [str(i) for i in range(100)]+['とき', 'なん', 'オンライン', '授業']
stop_words = ['こと', 'ため', 'よう', 'もの', 'これ', 'それ', 'どこ', 'そこ', 'たい', 'ほか', 'さっき', 'びと']+custom_stop_words

vectorizer = TfidfVectorizer(stop_words=stop_words)
X = vectorizer.fit_transform(df['tokens'])
df_vectorized = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

top_keywords = df_vectorized.sum().nlargest(50)
print(top_keywords)

In [ ]:
# ワードクラウドを作成してキーワードを可視化
!apt-get -y install fonts-ipafont-gothic
from matplotlib import pyplot as plt
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400, background_color='white',
            font_path='/usr/share/fonts/truetype/fonts-japanese-gothic.ttf').generate_from_frequencies(top_keywords)

plt.figure(figsize=(8, 4))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 各行の埋め込みを取得する
embeddings_opinion = model.encode(df['opinions'], convert_to_tensor=True)
df_embeddings_opinion = pd.DataFrame(embeddings_opinion)

# 以下は TF-IDF ベクトル化を実行していることが必要
# 行ごとにTF-IDFの高いキーワードを抽出して "-" でつなぎラベルにしておく
def get_top_keywords(row, top_n=3):
    sorted_row = row.sort_values(ascending=False)
    return '-'.join(sorted_row.index[:top_n])

df['keywords'] = df_vectorized.apply(get_top_keywords, axis=1)
df_embeddings_opinion.index = df['keywords']
df_embeddings_opinion.head()

In [ ]:
# 埋め込みにクラスター分析を適用してテキストを分類する
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
!pip install japanize-matplotlib
import japanize_matplotlib
import matplotlib.pyplot as plt

Z = linkage(df_embeddings_opinion, method="ward", metric="euclidean")
pd.DataFrame(Z)

fig, ax = plt.subplots(figsize=(10, 15))
dendrogram(Z, labels=df_embeddings_opinion.index, orientation="right", leaf_font_size=12)
plt.show()

In [ ]:
# t-SNE で埋め込みを次元削減して可視化
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=0)
tsne_results = tsne.fit_transform(df_embeddings_opinion)

# クラスター数に応じたカラーマップを作成
cluster_labels = fcluster(Z, t=4, criterion="maxclust")
unique_clusters = sorted(set(cluster_labels))
colors = plt.cm.tab10(range(1, len(unique_clusters)+1))
cluster_color_map = {cluster: color for cluster, color in zip(unique_clusters, colors)}

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.scatter(tsne_results[:, 0], tsne_results[:, 1], alpha=0)
for i, label in enumerate(df_embeddings_opinion.index):
  cluster = cluster_labels[i]
  color = cluster_color_map[cluster]
  plt.text(tsne_results[i, 0], tsne_results[i, 1], label, fontsize=9, color=color)
plt.show()